JOBSHEET 04:  
TEKNIK ANALISIS POSE & GEOMETRI TUBUH PADA GAMBAR 

Nama : Andrian Amirudin Rahmato
NIM : 4.33.25.0.02

Praktikum D1 - Inisialisasi Kamera dan Akuisisi Citra

In [1]:
import cv2, time 

cap = cv2.VideoCapture(0) 
if not cap.isOpened(): 
    raise RuntimeError("Kamera tidak bisa dibuka. Coba index 1/2.") 

frames, t0 = 0, time.time() 
while True: 
    ok, frame = cap.read() 
    if not ok: break 
    frames += 1 
    if time.time() - t0 >= 1.0: 
        cv2.setWindowTitle("Preview", f"Preview (FPS ~ {frames})") 
        frames, t0 = 0, time.time() 
    cv2.imshow("Preview", frame) 
    
    if cv2.waitKey(1) & 0xFF == ord('q'): break 

cap.release() 
cv2.destroyAllWindows() 

Praktikum D2 -- Deteksi Pose dan Analisis Sudut Tubuh

In [2]:
!pip install cvzone

In [3]:
import cv2, numpy as np 
from cvzone.PoseModule import PoseDetector 
  
cap = cv2.VideoCapture(0) 
if not cap.isOpened(): 
    raise RuntimeError("Kamera tidak bisa dibuka.") 
   
detector = PoseDetector(staticMode=False, modelComplexity=1, enableSegmentation=False, detectionCon=0.5, trackCon=0.5) 
  
while True: 
    # Tangkap setiap frame dari webcam 
    success, img = cap.read() 
  
    # Temukan pose manusia dalam frame 
    img = detector.findPose(img) 
  
    # Temukan landmark, bounding box, dan pusat tubuh dalam frame 
    # Set draw=True untuk menggambar landmark dan bounding box pada gambar 
    lmList, bboxInfo = detector.findPosition(img, draw=True, bboxWithHands=False) 
  
     # Periksa apakah ada landmark tubuh yang terdeteksi 
    if lmList: 
        # Dapatkan pusat bounding box di sekitar tubuh 
        center = bboxInfo["center"] 
  
        # Gambar lingkaran di pusat bounding box 
        cv2.circle(img, center, 5, (255, 0, 255), cv2.FILLED) 
  
        # Hitung jarak antara landmark 11 dan 15 dan gambarkan pada gambar 
        length, img, info = detector.findDistance(lmList[11][0:2], 
                                                     lmList[15][0:2], 
                                                     img=img, 
                                                     color=(255, 0, 0), 
                                                     scale=10) 
  
         # Hitung sudut antara landmark 11, 13, dan 15 dan gambarkan pada gambar 
        angle, img = detector.findAngle(lmList[11][0:2], 
                                         lmList[13][0:2], 
                                         lmList[15][0:2], 
                                         img=img, 
                                         color=(0, 0, 255), 
                                         scale=10) 
  
         
        # Periksa apakah sudut mendekati 50 derajat dengan offset 10 
        isCloseAngle50 = detector.angleCheck(myAngle=angle, 
                                                targetAngle=50, 
                                                offset=10) 
  
        # Cetak hasil pemeriksaan sudut         
        print(isCloseAngle50) 
  
   
    cv2.imshow("Pose + Angle ", img) 
    if cv2.waitKey(1) & 0xFF == ord('q'): break 
  
cap.release() 
cv2.destroyAllWindows()

d:\Kuliah Polines\MATKUL\Semester 3\3. Visi Komputer\Tugas\praktikum_4\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


Praktikum D3 — Deteksi Wajah dan Analisis Kedipan Mata

In [4]:
import cv2
import numpy as np
from cvzone.FaceMeshModule import FaceMeshDetector

# Indeks landmark mata kiri: vertikal (159, 145), horizontal (33, 133)
L_TOP, L_BOTTOM, L_LEFT, L_RIGHT = 159, 145, 33, 133

def dist(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka.")

# Inisialisasi objek FaceMeshDetector
detector = FaceMeshDetector(staticMode=False, maxFaces=2, 
                             minDetectionCon=0.5, minTrackCon=0.5)

# Variabel untuk penghitung kedipan
blink_count = 0
closed_frames = 0
CLOSED_FRAMES_THRESHOLD = 3   # Jumlah frame berturut-turut untuk dianggap kedipan
EYE_AR_THRESHOLD = 0.20      # Ambang EAR untuk menilai mata tertutup
is_closed = False

print("Memulai deteksi kedipan mata... Tekan 'q' pada jendela video untuk keluar.")

while True:
    ok, img = cap.read()
    if not ok: 
        break

    # Deteksi 468 landmark wajah
    img, faces = detector.findFaceMesh(img, draw=True)

    if faces:
        face = faces[0]  # List 468 titik (x, y) untuk wajah pertama
        
        # Hitung jarak vertikal dan horizontal mata kiri
        v = dist(face[L_TOP], face[L_BOTTOM])
        h = dist(face[L_LEFT], face[L_RIGHT])
        ear = v / (h + 1e-8)  # Eye Aspect Ratio (EAR)

        # Tampilkan nilai EAR di layar
        cv2.putText(img, f"EAR(L): {ear:.3f}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

        # Logika hitung kedipan: jika EAR < threshold selama beberapa frame
        if ear < EYE_AR_THRESHOLD:
            closed_frames += 1
            if closed_frames >= CLOSED_FRAMES_THRESHOLD and not is_closed:
                blink_count += 1
                is_closed = True
        else:
            closed_frames = 0
            is_closed = False

        # Tampilkan jumlah kedipan pada layar
        cv2.putText(img, f"Blink: {blink_count}", (20, 70), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow("FaceMesh + EAR", img)
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()

Memulai deteksi kedipan mata... Tekan 'q' pada jendela video untuk keluar.


Praktikum D4 — Deteksi Tangan dan Penghitungan Jumlah Jari 

In [5]:
import cv2
from cvzone.HandTrackingModule import HandDetector

# Inisialisasi webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka.")

# Inisialisasi HandDetector dari cvzone
# maxHands=1: Membatasi pendeteksian hanya untuk 1 tangan
detector = HandDetector(staticMode=False, maxHands=1, modelComplexity=1, 
                        detectionCon=0.5, minTrackCon=0.5)

print("Memulai deteksi tangan & jumlah jari... Tekan 'q' pada jendela video untuk keluar.")

while True:
    ok, img = cap.read()
    if not ok: 
        break

    # Deteksi tangan pada frame
    # flipType=True membalikkan orientasi visual agar seperti cermin
    hands, img = detector.findHands(img, draw=True, flipType=True)

    if hands:
        hand = hands[0]  # Mengambil data tangan pertama yang terdeteksi
        
        # fingersUp() mengembalikan list 5 elemen berupa 0 atau 1 [Ibu jari, Telunjuk, Tengah, Manis, Kelingking]
        fingers = detector.fingersUp(hand)
        count = sum(fingers)  # Menghitung total jari yang terangkat

        # Tampilkan jumlah jari dan status array [0/1] di layar
        cv2.putText(img, f"Fingers: {count}  {fingers}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow("Hands + Fingers", img)
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()

Memulai deteksi tangan & jumlah jari... Tekan 'q' pada jendela video untuk keluar.


Praktikum D5 — Pengenalan Gestur Tangan (Hand Gesture Recognition)

In [6]:
import cv2
import numpy as np
from cvzone.HandTrackingModule import HandDetector

# Fungsi untuk menghitung jarak Euclidean antara dua titik (x, y)
def dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# Fungsi klasifikasi gestur berdasarkan aturan geometri landmark tangan
def classify_gesture(hand):
    lm = hand["lmList"]  # 21 titik koordinat landmark tangan
    
    # Ambil posisi titik-titik utama
    wrist = np.array(lm[0][:2])        # Pergelangan tangan (0)
    thumb_tip = np.array(lm[4][:2])    # Ujung ibu jari (4)
    index_tip = np.array(lm[8][:2])    # Ujung jari telunjuk (8)
    middle_tip = np.array(lm[12][:2])  # Ujung jari tengah (12)
    ring_tip = np.array(lm[16][:2])    # Ujung jari manis (16)
    pinky_tip = np.array(lm[20][:2])   # Ujung jari kelingking (20)
    
    # Hitung rata-rata jarak semua ujung jari ke pergelangan tangan
    r_mean = np.mean([
        dist(index_tip, wrist),
        dist(middle_tip, wrist),
        dist(ring_tip, wrist),
        dist(pinky_tip, wrist),
        dist(thumb_tip, wrist)
    ])
    
    # --- ATURAN GESTUR ---
    # 1. Gestur "OK" (ujung ibu jari dan telunjuk saling menempel)
    if dist(thumb_tip, index_tip) < 35:
        return "OK"
        
    # 2. Gestur "THUMBS_UP" (ibu jari mengarah ke atas / koordinat Y lebih kecil)
    if (thumb_tip[1] < wrist[1] - 40) and (dist(thumb_tip, wrist) > 0.8 * dist(index_tip, wrist)):
        return "THUMBS_UP"
        
    # 3. Gestur "ROCK" (batu - tangan mengepal rapat)
    if r_mean < 120:
        return "ROCK"
        
    # 4. Gestur "PAPER" (kertas - semua jari terbuka lebar)
    if r_mean > 200:
        return "PAPER"
        
    # 5. Gestur "SCISSORS" (gunting - telunjuk dan jari tengah terbuka)
    if (dist(index_tip, wrist) > 180 and dist(middle_tip, wrist) > 180 and 
        dist(ring_tip, wrist) < 160 and dist(pinky_tip, wrist) < 160):
        return "SCISSORS"
        
    return "UNKNOWN"

# Inisialisasi webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka.")

detector = HandDetector(staticMode=False, maxHands=1, modelComplexity=1, 
                        detectionCon=0.5, minTrackCon=0.5)

print("Memulai pengenalan gestur... Tekan 'q' pada jendela video untuk keluar.")

while True:
    ok, img = cap.read()
    if not ok: 
        break

    hands, img = detector.findHands(img, draw=True, flipType=True)

    if hands:
        # Klasifikasikan gestur dari tangan yang terdeteksi
        label = classify_gesture(hands[0])
        
        # Tampilkan nama gestur di layar
        cv2.putText(img, f"Gesture: {label}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

    cv2.imshow("Hand Gestures (cvzone)", img)
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()

Memulai pengenalan gestur... Tekan 'q' pada jendela video untuk keluar.


Praktikum D6 — Analisis Gerakan Tubuh dan Penghitung Aktivitas

In [7]:
import cv2
import numpy as np
from collections import deque
from cvzone.PoseModule import PoseDetector

# Inisialisasi Mode & Parameter
MODE = "squat"           # Tekan tombol 'm' di jendela video untuk toggle ke "pushup"
KNEE_DOWN, KNEE_UP = 80, 160     # Ambang sudut lutut untuk squat (derajat)
DOWN_R, UP_R     = 0.85, 1.00    # Ambang rasio untuk push-up
SAMPLE_OK        = 4             # Jumlah frame konsisten untuk memvalidasi perubahan gerakan (debounce)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka.")

# Inisialisasi pendeteksi pose
detector = PoseDetector(staticMode=False, modelComplexity=1, 
                        enableSegmentation=False, detectionCon=0.5, trackCon=0.5)

count, state = 0, "up"
debounce = deque(maxlen=6)

# Fungsi menghitung rasio jarak untuk Push-Up (bahu-pergelangan / bahu-pinggul)
def ratio_pushup(lm):
    # Gunakan landmark tubuh sebelah kiri: 11=Shoulder, 15=Wrist, 23=Hip
    sh = np.array(lm[11][1:3])
    wr = np.array(lm[15][1:3])
    hp = np.array(lm[23][1:3])
    return np.linalg.norm(sh - wr) / (np.linalg.norm(sh - hp) + 1e-8)

print("Memulai Praktikum D6...")
print("-> Tekan 'm' untuk mengganti mode (Squat / Push-up)")
print("-> Tekan 'q' pada jendela video untuk keluar")

while True:
    ok, img = cap.read()
    if not ok: 
        break

    img = detector.findPose(img, draw=True)
    lmList, _ = detector.findPosition(img, draw=False)
    flag = None

    if lmList:
        if MODE == "squat":
            # Hitung sudut lutut kiri (hip:23, knee:25, ankle:27)
            angL, img = detector.findAngle(lmList[23][1:3],
                                            lmList[25][1:3],
                                            lmList[27][1:3],
                                            img=img, color=(0, 0, 255), scale=10)
             
            # Hitung sudut lutut kanan (hip:24, knee:26, ankle:28)
            angR, img = detector.findAngle(lmList[24][1:3],
                                            lmList[26][1:3],
                                            lmList[28][1:3],
                                            img=img, color=(0, 255, 0), scale=10)

            # Rata-rata sudut kedua lutut
            ang = (angL + angR) / 2.0
            
            if ang < KNEE_DOWN: 
                flag = "down"
            elif ang > KNEE_UP:  
                flag = "up"

            cv2.putText(img, f"Knee: {ang:5.1f} deg", (20, 70), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        else:  # Mode Push-Up
            r = ratio_pushup(lmList)
            if r < DOWN_R: 
                flag = "down"
            elif r > UP_R:  
                flag = "up"

            cv2.putText(img, f"Ratio: {r:4.2f}", (20, 70), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        # Logika Debounce untuk menghitung 1 repetisi penuh (DOWN -> UP)
        debounce.append(flag)
        if debounce.count("down") >= SAMPLE_OK and state == "up":
            state = "down"
        if debounce.count("up") >= SAMPLE_OK and state == "down":
            state = "up"
            count += 1  # Tambah hitungan repetisi

    # Tampilkan Informasi UI di Layar
    cv2.putText(img, f"Mode: {MODE.upper()}  Count: {count}", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    cv2.putText(img, f"State: {state}", (20, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow("Pose Counter", img)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): 
        break
    elif key == ord('m'):  # Toggle ganti mode antara squat dan pushup
        MODE = "pushup" if MODE == "squat" else "squat"
        count = 0  # Reset hitungan saat ganti mode

cap.release()
cv2.destroyAllWindows()

Memulai Praktikum D6...
-> Tekan 'm' untuk mengganti mode (Squat / Push-up)
-> Tekan 'q' pada jendela video untuk keluar
